In [ ]:
/kaggle/input/datasets/munnyvemula/sentidata1/training.1600000.processed.noemoticon.csv
/kaggle/input/datasets/munnyvemula/emojidata1/Emoji_Sentiment_Data_v1.0.csv

In [1]:
import pandas as pd
import numpy as np
import emoji
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
sent140 = pd.read_csv(
    "/kaggle/input/datasets/munnyvemula/sentidata1/training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None
)

sent140.columns = ["target", "id", "date", "query", "user", "text"]

sent140 = sent140[sent140["target"] != 2]
sent140["sentiment"] = sent140["target"].map({0: 0, 4: 2})

sent140 = sent140[["text", "sentiment"]]
sent140["text"] = sent140["text"].astype(str)

print("Sentiment140 size:", sent140.shape)

Sentiment140 size: (1600000, 2)


In [3]:
emoji_df = pd.read_csv(
    "/kaggle/input/datasets/munnyvemula/emojidata1/Emoji_Sentiment_Data_v1.0.csv"
)


In [4]:
emoji_list = set(emoji_df["Emoji"].values)

sent140["has_emoji"] = sent140["text"].apply(
    lambda x: any(c in emoji_list for c in x)
)

In [5]:
# Keep ONLY emoji-containing text
sent140 = sent140[sent140["has_emoji"] == True]

print("Final dataset size:", sent140.shape)
print(sent140["sentiment"].value_counts())

Final dataset size: (1409, 3)
sentiment
2    765
0    644
Name: count, dtype: int64


In [6]:
import re
import emoji

def preprocess(text):
    text = emoji.demojize(text)

    text = text.replace(":", " ")
    text = text.replace("_", " ")

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # remove mentions
    text = re.sub(r'@\w+', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.lower().strip()

In [7]:
import numpy as np

emoji_text_data = []

for _, row in emoji_df.iterrows():
    emoji_char = row["Emoji"]

    label = np.argmax([row["Negative"], row["Neutral"], row["Positive"]])

    emoji_text_data.append((f"{emoji_char}", label))
    emoji_text_data.append((f"I feel {emoji_char}", label))
    emoji_text_data.append((f"This is {emoji_char}", label))

emoji_extra = pd.DataFrame(emoji_text_data, columns=["text", "sentiment"])

# Apply preprocessing to new data
emoji_extra["text"] = emoji_extra["text"].apply(preprocess)

# Merge datasets
sent140 = pd.concat([sent140, emoji_extra])

In [8]:
from sklearn.utils import resample

df_neg = sent140[sent140["sentiment"] == 0]
df_neu = sent140[sent140["sentiment"] == 1]
df_pos = sent140[sent140["sentiment"] == 2]

max_size = max(len(df_neg), len(df_neu), len(df_pos))

df_neg = resample(df_neg, replace=True, n_samples=max_size, random_state=42)
df_neu = resample(df_neu, replace=True, n_samples=max_size, random_state=42)
df_pos = resample(df_pos, replace=True, n_samples=max_size, random_state=42)

sent140 = pd.concat([df_neg, df_neu, df_pos])

In [9]:
min_class = sent140["sentiment"].value_counts().min()

sent140 = sent140.groupby("sentiment").sample(
    n=min_class,
    random_state=42
)

print("Balanced dataset:")
print(sent140["sentiment"].value_counts())

Balanced dataset:
sentiment
0    2208
1    2208
2    2208
Name: count, dtype: int64


In [10]:
# First split: Train + Temp
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    sent140["text"].tolist(),
    sent140["sentiment"].tolist(),
    test_size=0.3,
    stratify=sent140["sentiment"],
    random_state=42
)

# Second split: Validation + Test
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    "vinai/bertweet-base",
    use_fast=False
)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset   = SentimentDataset(val_encodings, val_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    num_labels=3
)


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

In [14]:
training_args = TrainingArguments(
    output_dir="./bertweet_emoji",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=400,
    report_to="none",
)

In [15]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)


In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.796060,0.636821
2,0.854129,0.641908,0.749497
3,0.623663,0.499805,0.817907
4,0.623663,0.462771,0.834004


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1160, training_loss=0.6449920654296875, metrics={'train_runtime': 11809.7167, 'train_samples_per_second': 1.57, 'train_steps_per_second': 0.098, 'total_flos': 1219793804587008.0, 'train_loss': 0.6449920654296875, 'epoch': 4.0})

In [18]:
preds = trainer.predict(val_dataset)

y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("Validation Accuracy:", accuracy_score(y_true, y_pred))

# TEST ACCURACY
test_preds = trainer.predict(test_dataset)

y_test_pred = np.argmax(test_preds.predictions, axis=1)
y_test_true = test_preds.label_ids

test_accuracy = accuracy_score(y_test_true, y_test_pred)

print("Test Accuracy:", test_accuracy)

# Training accuracy (optional)
train_preds = trainer.predict(train_dataset)

train_true = train_preds.label_ids
train_pred = np.argmax(train_preds.predictions, axis=1)

train_accuracy = accuracy_score(train_true, train_pred)
print(f"Training Accuracy: {train_accuracy:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Validation Accuracy: 0.8340040241448692


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Accuracy: 0.8360160965794768


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Accuracy: 0.9060


In [19]:
def predict_sentiment(text):
    model.eval()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    text = preprocess(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    label = torch.argmax(outputs.logits, dim=1).item()

    return ["Negative", "Neutral", "Positive"][label]

In [20]:
tests = [
    "Great service 😒",
    "I love this phone 😍",
    "Worst update ever 😡",
    "Okay experience 😐",
    "Not bad 🙂"
]

for t in tests:
    print(t, "→", predict_sentiment(t))


Great service 😒 → Negative
I love this phone 😍 → Positive
Worst update ever 😡 → Negative
Okay experience 😐 → Negative
Not bad 🙂 → Positive
